In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/Dockerfile
FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y \
    git \
    curl \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .

RUN pip install --upgrade pip

RUN pip install --no-cache-dir \
    torch==2.3.1 \
    torchvision==0.18.1 \
    --index-url https://download.pytorch.org/whl/cpu

RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY .env.example .env

EXPOSE 8080

CMD exec uvicorn src.api.main:app --host 0.0.0.0 --port ${PORT:-8080}

Overwriting /content/drive/MyDrive/isic-flagship-project/Dockerfile


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/requirements.txt
fastapi==0.115.2
uvicorn[standard]==0.32.0
pydantic==2.9.2
pydantic-settings==2.6.1
sqlalchemy==2.0.35
alembic==1.13.2
aiosqlite==0.20.0
python-dotenv==1.0.1
pandas==2.2.2
numpy==1.26.4
pillow==10.4.0
timm==1.0.11
lightgbm==4.5.0
catboost==1.2.7
xgboost==2.1.1
scikit-learn==1.5.2
langchain==0.3.4
langchain-community==0.3.3
langchain-text-splitters==0.3.0
faiss-cpu==1.8.0.post1
sentence-transformers==3.1.1
python-multipart==0.0.9
httpx==0.27.2
structlog==24.4.0

Overwriting /content/drive/MyDrive/isic-flagship-project/requirements.txt


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/.dockerignore
__pycache__/
*.pyc
*.pyo
*.pyd
.Python
env/
venv/
.venv/
.git/
.gitignore
.ipynb_checkpoints/
*.ipynb
logs/

Overwriting /content/drive/MyDrive/isic-flagship-project/.dockerignore


In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
!gcloud projects list

PROJECT_ID              NAME                   PROJECT_NUMBER  ENVIRONMENT
isic-flagship-project   isic-flagship-project  918647643601
starry-arbor-463912-f8  My First Project       1045575864374


In [ ]:
PROJECT_ID = "isic-flagship-project"
REGION = "europe-west2"
SERVICE_NAME = "isic-api"

!gcloud config set project {PROJECT_ID}

Updated property [core/project].


In [ ]:
!gcloud services enable run.googleapis.com
!gcloud services enable cloudbuild.googleapis.com
!gcloud services enable artifactregistry.googleapis.com

In [ ]:
%cd /content/drive/MyDrive/isic-flagship-project

!gcloud run deploy {SERVICE_NAME} \
  --source . \
  --region {REGION} \
  --allow-unauthenticated \
  --memory 4Gi \
  --cpu 1 \
  --min-instances 1 \
  --max-instances 3 \
  --timeout 300 \
  --port 8080 \
  --cpu-boost

/content/drive/MyDrive/isic-flagship-project
Deploying from source requires an Artifact Registry Docker repository to store 
built containers. A repository named [cloud-run-source-deploy] in region 
[europe-west2] will be created.

Do you want to continue (Y/n)?  Y

Building using Dockerfile and deploying container to Cloud Run service [isic-api] in project [isic-flagship-project] region [europe-west2]
Service [isic-api] revision [isic-api-00001-srn] has been deployed and is serving 100 percent of traffic.
Service URL: https://isic-api-918647643601.europe-west2.run.app


In [ ]:
PROJECT_ID = "isic-flagship-project"
REGION = "europe-west2"
SERVICE_NAME = "isic-api"

!gcloud run services logs read {SERVICE_NAME} \
  --region {REGION} \
  --project {PROJECT_ID} \
  --limit 80

2026-05-07 18:59:45   warnings.warn(
2026-05-07 18:59:45 /usr/local/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_version" in HealthResponse has conflict with protected namespace "model_".
2026-05-07 18:59:45 You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
2026-05-07 18:59:45   warnings.warn(
2026-05-07 18:59:54 ✅ feature_engineering.py recreated successfully!
2026-05-07 18:59:54 Engine initialized on cpu
2026-05-07 18:59:55 /app/src/rag/rag_engine.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
2026-05-07 18:59:55   self.embeddings = HuggingFaceEmbeddings(
20